# Enterprise Agent with Agent Skills

This notebook runs a local autonomous coding agent that supports Agent Skills in directory format:
- `skills/<skill_name>/SKILL.md` (YAML frontmatter + Markdown instructions)
- optional `scripts/`, `references/`, and `assets/` inside each skill folder

CLI commands:
- `/reset`
- `/reload_skills`
- `/skills`
- `/skill-info <name>`
- `/activate-skill <name>`
- `/projects`
- `exit`

In [ ]:
import os

from core.agent import EnterpriseAgent
from core.llm import build_model
from core.memory import MemoryStore
from core.planner import Planner
from core.skill_manager import SkillManager
from core.tool_registry import ToolRegistry
from tools.csv_tool import CSVTool
from tools.filesystem_tool import FileSystemTool
from tools.python_tool import PythonTool
from tools.script_execution_tool import ScriptExecutionTool

In [ ]:
# Configuration
ROOT_DIR = os.getcwd()
PROJECTS_DIR = os.path.join(ROOT_DIR, "projects")
SKILLS_DIR = os.path.join(ROOT_DIR, "skills")
DB_PATH = os.path.join(ROOT_DIR, "data", "agent.db")
MAX_STEPS = 12

# Set your key before use
GIGACHAT_CREDENTIALS = "YOUR_AUTH_KEY"

os.makedirs(PROJECTS_DIR, exist_ok=True)
os.makedirs(SKILLS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

In [ ]:
# Build core components and register tools
model = build_model(GIGACHAT_CREDENTIALS)
memory = MemoryStore(DB_PATH)
planner = Planner()
registry = ToolRegistry()

fs_tool = FileSystemTool(ROOT_DIR)
py_tool = PythonTool()
csv_tool = CSVTool(ROOT_DIR)
script_tool = ScriptExecutionTool(ROOT_DIR)

registry.register("create_directory", fs_tool.create_directory, description="Create directory in projects/ or skills/")
registry.register("write_file", fs_tool.write_file, description="Write file under projects/ or skills/")
registry.register("read_file", fs_tool.read_file, description="Read file under projects/ or skills/")
registry.register("list_directory", fs_tool.list_directory, description="List directory under projects/ or skills/")
registry.register("run_python", py_tool.run_python, description="Execute Python code string")
registry.register("csv_preview", csv_tool.csv_preview, description="Preview rows from projects CSV file")
registry.register("csv_columns", csv_tool.csv_columns, description="Show columns from projects CSV file")
registry.register("run_skill_script", script_tool.run_script, description="Run a Python script under skills/*/scripts/")

skill_manager = SkillManager(SKILLS_DIR, registry, script_tool)

def reload_skills_tool(_input=""):
    script_tools = skill_manager.reload_skills()
    skill_names = [item["name"] for item in skill_manager.list_skills()]
    if not skill_names:
        return "No skills discovered."
    return "Skills: " + ", ".join(skill_names) + " | Script tools: " + ", ".join(script_tools)

def list_skills_tool(_input=""):
    return skill_manager.describe_skills()

def skill_info_tool(skill_name: str):
    return skill_manager.get_skill_info(skill_name)

registry.register("reload_skills", reload_skills_tool, description="Rescan skills directories and register script tools")
registry.register("list_skills", list_skills_tool, description="List installed skills and descriptions")
registry.register("skill_info", skill_info_tool, description="Load full SKILL.md for one skill")
registry.register("activate_skill", skill_manager.activate_skill, description="Activate skill instructions for current run")

skill_manager.reload_skills()
agent = EnterpriseAgent(model, memory, planner, registry, skill_manager, max_steps=MAX_STEPS)
print("Agent initialized with Agent Skills support.")

In [ ]:
def list_projects() -> str:
    names = []
    for item in sorted(os.listdir(PROJECTS_DIR)):
        path = os.path.join(PROJECTS_DIR, item)
        if os.path.isdir(path):
            names.append(item)
    return "\".join(names) if names else "(no projects)"


def parse_command(prefix: str, text: str) -> str:
    return text[len(prefix):].strip()


def run_cli() -> None:
    print("CLI started. Type 'exit' to stop.")
    while True:
        user_input = input(">> ").strip()

        if user_input == "exit":
            print("Stopping agent loop.")
            break

        if user_input == "/reset":
            memory.reset_conversation()
            print("Conversation memory reset.")
            continue

        if user_input == "/reload_skills":
            print(reload_skills_tool())
            continue

        if user_input == "/skills":
            print(skill_manager.describe_skills())
            continue

        if user_input.startswith("/skill-info "):
            target = parse_command("/skill-info ", user_input)
            try:
                print(skill_manager.get_skill_info(target))
            except Exception as exc:
                print(f"Error: {exc}")
            continue

        if user_input.startswith("/activate-skill "):
            target = parse_command("/activate-skill ", user_input)
            try:
                print(skill_manager.activate_skill(target))
            except Exception as exc:
                print(f"Error: {exc}")
            continue

        if user_input == "/projects":
            print(list_projects())
            continue

        if not user_input:
            continue

        result = agent.run(user_input)
        print(result)

In [ ]:
# Run interactive loop
run_cli()